## Setup

In [3]:
from pathlib import Path
import pandas as pd
import stringalign
from stringalign.evaluation import TranscriptionEvaluator

## Load data

In [4]:
data_path = Path("../data/")
trocr_path = data_path / "behandlet" / "trocr_predictions.csv"

In [5]:
trocr = pd.read_csv(trocr_path)

## Set up functions to automatically detect replacements

In [6]:
def make_evaluator_from_row(row: pd.Series, references_name="ground_truth", predictions_name="dan_prediction") -> TranscriptionEvaluator:
    return TranscriptionEvaluator.from_strings(
        references=[row[references_name]],
        predictions=[row[predictions_name]],
    )

def get_aggregated_confusion_matrix(evaluator: TranscriptionEvaluator) -> stringalign.statistics.StringConfusionMatrix:
    return sum(
        (
            stringalign.statistics.StringConfusionMatrix.from_strings_and_alignment(
                reference=le.reference, predicted=le.predicted, alignment=le.alignment
            )
            for le in evaluator.line_errors
        ),
        start=stringalign.statistics.StringConfusionMatrix.get_empty(),
    )


def get_edits(confusion_matrix: stringalign.statistics.StringConfusionMatrix) -> dict[str, tuple[tuple[str, str], int]]:
    return tuple(
        ((op.generalize().substring, op.generalize().replacement), count)
        for op, count in confusion_matrix.edit_counts.most_common()
    )

def get_true_positives(confusion_matrix: stringalign.statistics.StringConfusionMatrix) -> dict[str, tuple[str, int]]:
    return tuple(confusion_matrix.true_positives.items())

def get_false_positives(confusion_matrix: stringalign.statistics.StringConfusionMatrix) -> dict[str, tuple[str, int]]:
    return tuple(confusion_matrix.false_positives.items())

def get_false_negatives(confusion_matrix: stringalign.statistics.StringConfusionMatrix) -> dict[str, tuple[str, int]]:
    return tuple(confusion_matrix.false_negatives.items())

## Compute replacements

We use optimal string alignment to automatically detect character replacements. For example, if we have the two strings:

`string`  
`align`

where `string` is the "ground truth" and `align` is the "prediction. An optimal alignment of these two strings could be

`string-`  
`al-i-gn`

which is equivalent to the following operations:

* `Replace('a', 's')`
* `Replace('l', 't')`
* `Insert('r')`
* `Keep('i')`
* `Insert('n')`
* `Keep('g')`
* `Delete('n')`

From this, we can compute statistics on how common different character replacements are. However, the strings `string` and `align` have other alignments that are just as valid as well. Take for example:

`string-`  
`-ali-gn`

* `Insert('s')`
* `Replace('a', 't')`
* `Replace('l', 'r')`
* `Keep('i')`
* `Insert('n')`
* `Keep('g')`
* `Delete('n')`

To mitigate this somewhat, we can aggregate the replacements:

* `Replace('al', 'str')`
* `Keep('i')`
* `Insert('n')`
* `Keep('g')`
* `Delete('n')`

This will give much more stable alignments (but there may still be some strings that have multiple possible alignments, particularly when letters swap places.

## Types of error statistics:

We compute four types of error statistics: "edits", "true positives", "false positives" and "false negatives".

### Edits
A edit, `('e', 'a'), 3`, should be interpreted as there were three times the letter 'a' was transcribed as the letter 'e'.

### True positives
A true positive, `'a', 10`, means that there were ten times the letter 'a' was correctly transcribed

### False positives
A false positive, ('a', 2) means that there were two times the letter 'a' was transcribed when something else should have been transcribed

### False negatives
A false negatives, ('a', 5) means that there were five times the letter 'a' should have been transcribed, but something else was transcribed.

## Example
The alignment operations `[Replace('al', 'str'), Keep('i'), Insert('n'), Keep('g'), Delete('n')` have the following error statistics:

* **Edits**: `('al', 'str'), 1`, `('n', ''), 1`, `('', 'n')`
* **True positives**: `('i', 1)`, `('g', 1)`
* **False positives**: `('al', 1)`, `('n', 1)`
* **False negatives**: `('str', 1)`, `('n', 1)`

In [7]:
dfs = {"trocr": trocr}
aggregated_confusion_matrices = {}

for model, df in dfs.items():
    df["evaluators"] = df.apply(
        make_evaluator_from_row, axis=1, references_name="text", predictions_name=f"predictions"
    )
    df["aggregated_confusion_matrices"] = df["evaluators"].map(get_aggregated_confusion_matrix)
    
    df["edits"] = df["aggregated_confusion_matrices"].map(get_edits)
    df["true_positives"] = df["aggregated_confusion_matrices"].map(get_true_positives)
    df["false_positives"] = df["aggregated_confusion_matrices"].map(get_false_positives)
    df["false_negatives"] = df["aggregated_confusion_matrices"].map(get_false_negatives)
    
    aggregated_confusion_matrices[model] = sum(
        (cm for cm in df["aggregated_confusion_matrices"]),
        start=stringalign.statistics.StringConfusionMatrix.get_empty(),
    )

## Look for hallucinations
(By finding contiguous errors longer than half the text size)

In [8]:
for document in trocr.itertuples():
    if any(len(edit[0][0]) > 0.5 * len(document.text) for edit in document.edits):
        print("===")
        print("\ntruth")
        print("-----")
        print(document.text)
        print("\nprediction")
        print("----------")
        print(document.predictions)
        print("\nedits")
        print("--------")
        print(document.edits)

===

truth
-----
for

prediction
----------
jeg

edits
--------
((('jeg', 'for'), 1),)
===

truth
-----
Vi

prediction
----------
Være

edits
--------
((('ære', 'i'), 1),)
===

truth
-----
H Mohn.

prediction
----------
Helvbr.

edits
--------
((('elvbr', ' Mohn'), 1),)
===

truth
-----
FRA

prediction
----------
Faa

edits
--------
((('aa', 'RA'), 1),)
===

truth
-----
Bsv -

prediction
----------
Bjørn-

edits
--------
((('jørn', 'sv '), 1),)


All cases where a contiguous error was at least 50% the length of the textline occured for short edits that occured in very short text lines. This shows that the TrOCR model didn't hallucinate long strings of text at least.

## Row-wise data

In [9]:
trocr

,text,predictions,evaluators,aggregated_confusion_matrices,edits,true_positives,false_positives,false_negatives
0,Til Bestyrelsen af,Til Bestyrelsen af,TranscriptionEvaluator(references=('Til Bestyr...,StringConfusionMatrix(true_positives=Counter({...,(),"((T, 1), (i, 1), (l, 2), ( , 2), (B, 1), (e, 3...",(),()
1,I Ærbødighed,I Ærbødighed,TranscriptionEvaluator(references=('I Ærbødigh...,StringConfusionMatrix(true_positives=Counter({...,(),"((I, 1), ( , 1), (Æ, 1), (r, 1), (b, 1), (ø, 1...",(),()
2,Paa Søskendes og egne Vegne,Paa Søskendas og egne Vegne,TranscriptionEvaluator(references=('Paa Søsken...,StringConfusionMatrix(true_positives=Counter({...,"(((a, e), 1),)","((P, 1), (a, 2), ( , 4), (S, 1), (ø, 1), (s, 2...","((a, 1),)","((e, 1),)"
3,14/7-08.,14/7-08.,"TranscriptionEvaluator(references=('14/7-08.',...",StringConfusionMatrix(true_positives=Counter({...,(),"((1, 1), (4, 1), (/, 1), (7, 1), (-, 1), (0, 1...",(),()
4,"Granskog, Hvalstad.","granskog, Hvalstad.","TranscriptionEvaluator(references=('Granskog, ...",StringConfusionMatrix(true_positives=Counter({...,"(((g, G), 1),)","((r, 1), (a, 3), (n, 1), (s, 2), (k, 1), (o, 1...","((g, 1),)","((G, 1),)"
...,...,...,...,...,...,...,...,...
1557,1875,1875,"TranscriptionEvaluator(references=('1875',), p...",StringConfusionMatrix(true_positives=Counter({...,(),"((1, 1), (8, 1), (7, 1), (5, 1))",(),()
1558,forgjæves. -,forgjæres. -,TranscriptionEvaluator(references=('forgjæves....,StringConfusionMatrix(true_positives=Counter({...,"(((r, v), 1),)","((f, 1), (o, 1), (r, 1), (g, 1), (j, 1), (æ, 1...","((r, 1),)","((v, 1),)"
1559,G. Neergaard,G. Neergaard,TranscriptionEvaluator(references=('G. Neergaa...,StringConfusionMatrix(true_positives=Counter({...,(),"((G, 1), (., 1), ( , 1), (N, 1), (e, 2), (r, 2...",(),()
1560,Indk 14/6 1882,Inda 14/6 1882,TranscriptionEvaluator(references=('Indk 14/6 ...,StringConfusionMatrix(true_positives=Counter({...,"(((a, k), 1),)","((I, 1), (n, 1), (d, 1), ( , 2), (1, 2), (4, 1...","((a, 1),)","((k, 1),)"


## Most common edits overall (aggregated)

In [10]:
for op, count in aggregated_confusion_matrices["trocr"].edit_counts.most_common(50):
    print(op, count)

Insert(substring='"') 36
Insert(substring=' ') 31
Insert(substring='e') 29
Insert(substring='s') 26
Insert(substring='.') 25
Delete(substring='.') 21
Replace(substring='e', replacement='a') 20
Insert(substring=',') 19
Replace(substring='a', replacement='o') 19
Replace(substring='e', replacement='i') 18
Insert(substring='r') 18
Delete(substring='e') 16
Delete(substring=',') 16
Delete(substring=' ') 15
Replace(substring='i', replacement='e') 14
Replace(substring='l', replacement='t') 13
Delete(substring='r') 13
Replace(substring='o', replacement='a') 13
Replace(substring='r', replacement='s') 12
Replace(substring='o', replacement='e') 12
Insert(substring='n') 12
Replace(substring='d', replacement='s') 12
Delete(substring='s') 11
Replace(substring='a', replacement='e') 10
Replace(substring='n', replacement='r') 10
Replace(substring='.', replacement='-') 10
Replace(substring='g', replacement='j') 10
Replace(substring='r', replacement='n') 10
Replace(substring='d', replacement='D') 10
Repla

## Save aggregated errors

In [12]:
import json
for model, confusion_matrix in aggregated_confusion_matrices.items():
    data = {
        "edits": get_edits(confusion_matrix),
        "true_positives": get_true_positives(confusion_matrix),
        "false_positives": get_false_positives(confusion_matrix),
        "false_negatives": get_false_negatives(confusion_matrix),
    }
    Path(f"aggregated_errors_{model}.json").write_text(json.dumps(data))